In [ ]:
import psycopg2
import pandas as pd
import geopandas as gpd
from geopandas import GeoSeries, GeoDataFrame

import math
import numpy as np
import os
import time
import gc
import glob as gb

from tqdm import tqdm
from shapely import wkt
from shapely import geometry
from shapely.geometry import Polygon, Point , MultiPoint, LineString, MultiPolygon
from shapely.ops import unary_union

import matplotlib.pyplot as plt
import networkx as nx
import itertools

# CSV 파일 읽기
df = pd.read_csv("C:/Users/kfq/Desktop/P-MEDIAN 알고리즘/지표별 점수화.csv")

# 'geometry' 열이 WKT 문자열로 저장되어 있으므로 이를 Shapely 객체로 변환
df['geometry'] = df['geometry'].apply(wkt.loads)

df = gpd.GeoDataFrame(df, crs='epsg:4326')

# 각 폴리곤의 중심 계산
df['center'] = df['geometry'].centroid

df[['접근성', '수요성', '통행량', '가용성', '활동성']] = df[['접근성', '수요성', '통행량', '가용성', '활동성']].astype('float')

# 그래프 생성
G = nx.Graph()

# 위치 딕셔너리 만들기
locations = df['center'].to_dict()

# 지표 속성 딕셔너리 만들기
demand_factors = df[['접근성', '수요성', '통행량', '가용성', '활동성']].to_dict(orient='index')

# 지표별 가중치 부여
weight_factors = {'접근성': 1.4, '수요성': 1.2, '통행량': 0.6, '가용성': 1.2, '활동성': 0.6}

# 노드 및 지표값 추가
for index, center in locations.items():
    lat = locations[index].x
    lon = locations[index].y
    G.add_node(index, latitude=lat, longitude=lon, **demand_factors[index])

# 노드 간의 거리 계산 및 엣지 추가
for node1, node2 in itertools.combinations(G.nodes, 2):
    lat1, lon1 = G.nodes[node1]["latitude"], G.nodes[node1]["longitude"]
    lat2, lon2 = G.nodes[node2]["latitude"], G.nodes[node2]["longitude"]
    distance = ((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5  # 유클리드 거리 계산
    G.add_edge(node1, node2, weight=distance)


In [ ]:
import itertools
import pandas as pd
import networkx as nx

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index
p = 2

# 결과를 저장할 리스트
results = []

# 모든 가능한 노드 조합 탐색
for combination in itertools.combinations(parking_nodes, p):
    # 각 지표별 가중치 거리 합을 저장할 딕셔너리 초기화
    factor_weighted_sums = {factor: 0 for factor in weight_factors}
    
    for node in G.nodes:
        # 각 노드와 가장 가까운 시설까지의 거리 계산
        min_distance = min(G[node][facility]['weight'] if node != facility else 0 for facility in combination)
        
        # 각 지표별 가중치와 거리를 곱하여 거리 합산
        for factor in weight_factors:
            factor_weighted_sums[factor] += G.nodes[node][factor] * weight_factors[factor] * min_distance
    
    # 리스트
    results.append({
        'Combination': combination,
        **factor_weighted_sums
    })

df_results = pd.DataFrame(results)

print(df_results)

output_file_path = "C:/Users/kfq/Desktop/combination_results_2_me.csv"
df_results.to_csv(output_file_path, index=False, encoding='utf-8-sig')

print(f"결과가 CSV 파일로 저장되었습니다: {output_file_path}")

In [ ]:
output_file_path = "C:/Users/kfq/Desktop/combination_results.csv"
df_results.to_csv(output_file_path, index=False, encoding='utf-8-sig')

print(f"결과가 CSV 파일로 저장되었습니다: {output_file_path}")

In [ ]:
import itertools
import pandas as pd
import networkx as nx

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index
p = 2

# 결과를 저장할 리스트
results = []

# 모든 가능한 노드 조합 탐색
for combination in itertools.combinations(parking_nodes, p):
    # 각 지표별 가중치 거리 합을 저장할 딕셔너리 초기화
    factor_weighted_sums = {factor: 0 for factor in weight_factors}
    for node in G.nodes:
        # 각 노드와 가장 가까운 시설까지의 거리 계산
        min_distance = min(G[node][facility]['weight'] if node != facility else 0 for facility in combination)
        
        # 각 지표별 가중치와 거리를 곱하여 거리 합산
        for factor in weight_factors:
            factor_weighted_sums[factor] += G.nodes[node][factor] * weight_factors[factor] * min_distance
    
    # 결과를 리스트에 추가
    results.append({
        'Combination': combination,
        **factor_weighted_sums
    })

# 결과를 데이터프레임으로 변환
df_results2 = pd.DataFrame(results)

# 결과 출력
print(df_results2)


In [ ]:
output_file_path = "C:/Users/kfq/Desktop/combination_results2.csv"
df_results2.to_csv(output_file_path, index=False, encoding='utf-8-sig')

print(f"결과가 CSV 파일로 저장되었습니다: {output_file_path}")

In [ ]:
import psycopg2
import pandas as pd
import geopandas as gpd
from geopandas import GeoSeries, GeoDataFrame

import math
import numpy as np
import os
import time
import gc
import glob as gb

from tqdm import tqdm
from shapely import wkt
from shapely import geometry
from shapely.geometry import Polygon, Point , MultiPoint, LineString, MultiPolygon
from shapely.ops import unary_union

import matplotlib.pyplot as plt
import networkx as nx
import itertools

# CSV 파일 읽기
df = pd.read_csv("C:/Users/kfq/Desktop/P-MEDIAN 알고리즘/범주화 데이터.csv")

# 'geometry' 열이 WKT 문자열로 저장되어 있으므로 이를 Shapely 객체로 변환
df['geometry'] = df['geometry'].apply(wkt.loads)

df = gpd.GeoDataFrame(df, crs='epsg:4326')

# 각 폴리곤의 중심 계산
df['center'] = df['geometry'].centroid

df[['접근성', '수요성', '통행량', '가용성', '활동성']] = df[['접근성', '수요성', '통행량', '가용성', '활동성']].astype('float')

# 그래프 생성
G = nx.Graph()

# 위치 딕셔너리 만들기
locations = df['center'].to_dict()

# 지표 속성 딕셔너리 만들기
demand_factors = df[['접근성', '수요성', '통행량', '가용성', '활동성']].to_dict(orient='index')

# 지표별 가중치 부여
weight_factors = {'접근성': 1, '수요성': 1, '통행량': 1, '가용성': 1, '활동성': 1}

# 노드 및 지표값 추가
for index, center in locations.items():
    lat = locations[index].x
    lon = locations[index].y
    G.add_node(index, latitude=lat, longitude=lon, **demand_factors[index])

# 노드 간의 거리 계산 및 엣지 추가
for node1, node2 in itertools.combinations(G.nodes, 2):
    lat1, lon1 = G.nodes[node1]["latitude"], G.nodes[node1]["longitude"]
    lat2, lon2 = G.nodes[node2]["latitude"], G.nodes[node2]["longitude"]
    distance = ((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5  # 유클리드 거리 계산
    G.add_edge(node1, node2, weight=distance)

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index
p = 2

# 4칸 이내의 이웃 후보지를 찾기 위한 거리 임계값 설정
# 이 값은 실제 격자의 크기에 따라 다를 수 있습니다. 이 예시에서는 4칸 이내의 거리로 설정합니다.
distance_threshold = 4 * ((df['center'].x.max() - df['center'].x.min()) / math.sqrt(len(df)))

# 후보지의 이웃 찾기
neighbor_candidates = {}
for node in parking_nodes:
    neighbors = []
    for other_node in parking_nodes:
        if node != other_node and G[node][other_node]['weight'] <= distance_threshold:
            neighbors.append(other_node)
    neighbor_candidates[node] = neighbors

# 결과를 저장할 리스트
results = []

# 각 후보지에서 4칸 이내의 이웃과의 조합을 생성
for node in parking_nodes:
    neighbors = neighbor_candidates[node]
    if len(neighbors) >= p-1:  # 최소 p-1 개의 이웃이 있어야 조합을 만들 수 있음
        for combination in itertools.combinations(neighbors, p-1):
            full_combination = (node,) + combination
            # 각 지표별 가중치 거리 합을 저장할 딕셔너리 초기화
            factor_weighted_sums = {factor: 0 for factor in weight_factors}
            
            for other_node in G.nodes:
                # 각 노드와 가장 가까운 시설까지의 거리 계산
                min_distance = min(G[other_node][facility]['weight'] if other_node != facility else 0 for facility in full_combination)
                
                # 각 지표별 가중치와 거리를 곱하여 거리 합산
                for factor in weight_factors:
                    factor_weighted_sums[factor] += G.nodes[other_node][factor] * weight_factors[factor] * min_distance
            
            # 결과 저장
            results.append({
                'Combination': full_combination,
                **factor_weighted_sums
            })

df_results = pd.DataFrame(results)

output_file_path = "C:/Users/kfq/Desktop/4_result.csv"
df_results.to_csv(output_file_path, index=False, encoding='utf-8-sig')

print(f"결과가 CSV 파일로 저장되었습니다: {output_file_path}")


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

# 기본 지도 시각화
fig, ax = plt.subplots(figsize=(15, 15))
df.plot(ax=ax, color='lightgrey', edgecolor='black')

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index

# 그래프 시각화
pos = {node: (G.nodes[node]['latitude'], G.nodes[node]['longitude']) for node in G.nodes()}

# 중심 노드 (주차장이 있는 노드) 시각화
nx.draw_networkx_nodes(G, pos, nodelist=parking_nodes, node_size=10, node_color='red', ax=ax)

# 특정 거리 내의 노드만 연결하도록 필터링
threshold_distance = 0.015# 거리 기준 설정 (예: 유클리드 거리)

# 거리 기준에 맞는 엣지들만 시각화
for node in parking_nodes:
    neighbors = [
        neighbor for neighbor in G.nodes 
        if G.nodes[node] != G.nodes[neighbor] and 
           ((G.nodes[node]['latitude'] - G.nodes[neighbor]['latitude'])**2 + 
            (G.nodes[node]['longitude'] - G.nodes[neighbor]['longitude'])**2)**0.5 <= threshold_distance
    ]
    nx.draw_networkx_edges(G, pos, edgelist=[(node, neighbor) for neighbor in neighbors], edge_color='red', ax=ax)

# 주변 노드 (연결된 노드) 시각화
surrounding_nodes = set(itertools.chain(*[
    neighbors for node in parking_nodes
]))
nx.draw_networkx_nodes(G, pos, nodelist=surrounding_nodes, node_size=10, node_color='black', ax=ax)

plt.title('Spatial Network Graph with Local Edges')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point
import itertools

# 기본 지도 시각화
fig, ax = plt.subplots(figsize=(15, 15))
df.plot(ax=ax, color='lightgrey', edgecolor='black')

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index

# 그래프 시각화
pos = {node: (G.nodes[node]['latitude'], G.nodes[node]['longitude']) for node in G.nodes()}

# 중심 노드 (주차장이 있는 노드) 시각화
nx.draw_networkx_nodes(G, pos, nodelist=parking_nodes, node_size=30, node_color='red', ax=ax)

# 특정 거리 내의 노드만 연결하도록 필터링
threshold_distance = 0.015  # 거리 기준 설정 (예: 유클리드 거리)

# 거리 기준에 맞는 엣지들만 시각화 및 주변 노드 생성
surrounding_nodes = set()  # 주변 노드 집합 초기화
for node in parking_nodes:
    neighbors = [
        neighbor for neighbor in G.nodes 
        if G.nodes[node] != G.nodes[neighbor] and 
           ((G.nodes[node]['latitude'] - G.nodes[neighbor]['latitude'])**2 + 
            (G.nodes[node]['longitude'] - G.nodes[neighbor]['longitude'])**2)**0.5 <= threshold_distance
    ]
    # 엣지 시각화
    nx.draw_networkx_edges(G, pos, edgelist=[(node, neighbor) for neighbor in neighbors], edge_color='red', ax=ax)
    
    # 주변 노드 집합에 추가
    surrounding_nodes.update(neighbors)

# 주변 노드 시각화
nx.draw_networkx_nodes(G, pos, nodelist=surrounding_nodes, node_size=10, node_color='black', ax=ax)

plt.title('Spatial Network Graph with Local Edges')
plt.show()


In [ ]:
import random
import matplotlib.pyplot as plt
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

# 기본 지도 시각화
fig, ax = plt.subplots(figsize=(15, 15))
df.plot(ax=ax, color='lightgrey', edgecolor='black')

# 공영주차장이 0이 아닌 노드만 선택
parking_nodes = df[df['공영주차장'] != 0].index

# 그래프 시각화
pos = {node: (G.nodes[node]['latitude'], G.nodes[node]['longitude']) for node in G.nodes()}

# 중심 노드 (주차장이 있는 노드) 시각화
nx.draw_networkx_nodes(G, pos, nodelist=parking_nodes, node_size=30, node_color='red', ax=ax)

# 거리 기준에 맞는 엣지들만 시각화 및 주변 노드 생성
surrounding_nodes = set()  # 주변 노드 집합 초기화

# Introduce randomness in the threshold distance
min_distance = 0.010  # 최소 거리
max_distance = 0.025  # 최대 거리

# Edge number settings: Random number of edges for each node
min_edges = 5  # 최소 엣지 개수
max_edges = 20  # 최대 엣지 개수

for node in parking_nodes:
    # Apply a random threshold distance for each node
    threshold_distance = random.uniform(min_distance, max_distance)
    
    # 해당 거리 안에 있는 이웃 노드 찾기
    potential_neighbors = [
        neighbor for neighbor in G.nodes 
        if G.nodes[node] != G.nodes[neighbor] and 
           ((G.nodes[node]['latitude'] - G.nodes[neighbor]['latitude'])**2 + 
            (G.nodes[node]['longitude'] - G.nodes[neighbor]['longitude'])**2)**0.5 <= threshold_distance
    ]
    
    # 엣지의 개수를 랜덤하게 결정 (min_edges와 max_edges 사이)
    num_edges = random.randint(min_edges, max_edges)
    
    # 이웃 노드 중에서 랜덤으로 num_edges만큼 선택
    neighbors = random.sample(potential_neighbors, min(len(potential_neighbors), num_edges))
    
    # 엣지 시각화 (draw edges between the node and its randomly selected neighbors)
    nx.draw_networkx_edges(G, pos, edgelist=[(node, neighbor) for neighbor in neighbors], edge_color='red', ax=ax)
    
    # 주변 노드 집합에 추가 (add neighbors to the surrounding nodes set)
    surrounding_nodes.update(neighbors)

# 주변 노드 시각화 (draw surrounding nodes)
nx.draw_networkx_nodes(G, pos, nodelist=surrounding_nodes, node_size=10, node_color='black', ax=ax)

plt.title('Spatial Network Graph with Randomized Edges and Neighbors')
plt.show()
